# Phase 1 - Architecture 1: LLaMA-style decoder-only Transformer

**Baseline:** RMSNorm + RoPE + SwiGLU + MHA, untied embeddings.  
**Modified:** GQA + QK-norm + tied embeddings + value residual + multi-token prediction + source embedding (Act/HC/SC/Tribunal), depth 12 -> 17 to re-spend saved parameters.

**Fast generation:** `model.prefill()` reads the prompt once, then `model.step()` feeds one token per step (KV cache; recurrent state for linear-attention layers; encoder run once for the encoder-decoder), instead of re-running the whole prefix for every new token.

**Run order:** run all cells top to bottom. Change `VARIANT` (`'baseline'` / `'modified'`) or `OVERRIDES` in the config cell for ablations; the notebook writes `results.json`, the loss plot, `prediction.json` and `submission.zip` to its own output folder.

Attach the `kratikasum/tokenizer` Kaggle dataset as an input, and add your Hugging Face token as the Kaggle secret `HF`.

## 1. Configuration

### Imports, Hugging Face login, GPU check

In [2]:
import os, random, json, time, math, zipfile, gc, glob, pickle
from array import array
from dataclasses import asdict
from typing import List, Dict, Tuple
import numpy as np
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CUDA available: True
GPU 0: Tesla T4
GPU 1: Tesla T4


### Data

In [3]:
# ---------------- data ----------------
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
TEST_FILES = ['Phase_1_Pretraining/Test/test.json', 'Phase_1_Pretraining/Test/Phase1- Test.json']  # first one that exists
TRAIN_TOKENS = 400_000_000
VAL_TOKENS = 10_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

### Model choice

In [4]:
# ---------------- model choice ----------------
VARIANT = 'modified'        # 'baseline' or 'modified'
OVERRIDES = {}              # ablations, e.g. {'qk_norm': False}
RUN_TAG = ''                # give every ablation run its own tag, e.g. '_noqknorm', so runs don't share a resume.pt
GRAD_CKPT = True            # saves memory (~30% slower). Set False if you have memory to spare.

### Training

In [5]:
# ---------------- training ----------------
N_GPU = torch.cuda.device_count()                  # Kaggle T4 x2 -> 2: every batch is split across both GPUs
PER_GPU_BATCH = 4           # sequences per GPU per forward pass (lower this if you get CUDA OOM)
MICRO_BATCH = PER_GPU_BATCH * max(1, N_GPU)
GRAD_ACCUM = max(1, 128 // MICRO_BATCH)             # effective batch stays 16 sequences (= 32k tokens) on 1 or 2 GPUs
LR = 6e-4
MIN_LR = LR * 0.1
WARMUP_STEPS = 300
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
EVAL_EVERY = 250            # optimizer steps
EVAL_BLOCKS = 128           # validation sequences used during training (full val set used at the end)
LOG_EVERY = 25
SAVE_EVERY = 250            # save a full resume checkpoint (model + optimizer + progress) every N steps
MAX_TRAIN_MINUTES = 720     # wall-clock cap: the LR schedule is stretched so it always finishes decaying

### Generation

In [6]:
# ---------------- generation ----------------
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.3
TOP_P = 0.85
TOP_K = 30
REPETITION_PENALTY = 1.1    # 1.0 = off (>1 also penalises every token already in the 512-word prompt)
GEN_BATCH = 32            # sequences generated together (the KV cache is small, so 32-64 is fine)

### Output paths

In [7]:
OUTPUT_DIR = f'/kaggle/working/arch1_decoder_{VARIANT}{RUN_TAG}'   # one folder (and one resume.pt) per run
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'legal-final')
PREDICTION_PATH = os.path.join(OUTPUT_DIR, 'prediction.json')
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, 'submission.zip')
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, 'training_validation_loss.png')
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

### Device and precision

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# T4/P100 have no bf16 -> fp16 + GradScaler; Ampere+ -> bf16
use_bf16 = device == 'cuda' and torch.cuda.get_device_capability()[0] >= 8
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print('Configuration loaded. device =', device, '| amp dtype =', amp_dtype)

Configuration loaded. device = cuda | amp dtype = torch.float16


## 2. Load tokenizer

### Tokenizer class

In [9]:
class ByteLevelBPE:
    """Your tokenizer class (training code removed: we load the already-trained pickle)."""
    def __init__(self):
        self.vocab: Dict[bytes, int] = {}
        self.inverse_vocab: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], Tuple[bytes, int]] = {}
        self.word_end_marker = b'</w>'
        self.reserved_tokens = [b'<pad>', b'<unk>', b'<bos>', b'<eos>']

    def tokenize(self, text: str) -> List[bytes]:
        words = text.split()
        tokens = []
        for w in words:
            word_bytes = w.encode('utf-8')
            split = [bytes([b]) for b in word_bytes] + [self.word_end_marker]
            while True:
                best_pair, best_rank = None, float('inf')
                for i in range(len(split) - 1):
                    pair = (split[i], split[i+1])
                    if pair in self.merges:
                        rank = self.merges[pair][1]
                        if rank < best_rank:
                            best_rank, best_pair = rank, pair
                if best_pair is None:
                    break
                new_split, i = [], 0
                while i < len(split):
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i+1] == best_pair[1]:
                        new_split.append(self.merges[best_pair][0])
                        i += 2
                    else:
                        new_split.append(split[i])
                        i += 1
                split = new_split
            tokens.extend(split)
        return tokens

    def encode(self, text: str) -> List[int]:
        unk_id = self.vocab[b'<unk>']
        return [self.vocab.get(token, unk_id) for token in self.tokenize(text)]

    def decode(self, token_ids: List[int]) -> str:
        skip_ids = {self.vocab[t] for t in self.reserved_tokens if t in self.vocab}
        bytes_list = []
        for tid in token_ids:
            if tid in skip_ids:
                continue
            bytes_list.append(self.inverse_vocab.get(tid, b''))
        joined = b''.join(bytes_list)
        return joined.replace(self.word_end_marker, b' ').decode('utf-8', errors='replace').strip()

    @classmethod
    def load(cls, filepath: str):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        bpe = cls()
        bpe.vocab = data['vocab']
        bpe.inverse_vocab = {v: k for k, v in bpe.vocab.items()}
        bpe.merges = data['merges']
        return bpe

    def __len__(self):
        return len(self.vocab)

    @property
    def eos_token_id(self):
        return self.vocab.get(b'<eos>')

### Load the trained tokenizer, check vocabulary size

In [10]:
cands = glob.glob('/kaggle/input/**/bpe_tokenized_txt.pkl', recursive=True)
if cands:
    tok_path = cands[0]
else:                                   # dataset not attached as input -> download it
    import kagglehub
    tok_path = os.path.join(kagglehub.dataset_download('kratikasum/tokenizer'), 'bpe_tokenized_txt.pkl')
tokenizer = ByteLevelBPE.load(tok_path)

# The embedding table must cover the largest id, and ids should be unique.
VOCAB_SIZE = max(tokenizer.vocab.values()) + 1
n_unique = len(set(tokenizer.vocab.values()))
print(f'tokenizer entries: {len(tokenizer):,} | unique ids: {n_unique:,} | embedding size: {VOCAB_SIZE:,}')
if n_unique != len(tokenizer):
    print('WARNING: some tokens share an id (can happen when two merges build the same byte string).')

tokenizer entries: 48,000 | unique ids: 48,000 | embedding size: 48,000


### Quick encode / decode test

In [11]:
ids = tokenizer.encode("The Supreme Court dismissed the appeal filed by the petitioner.")
print(ids); print(tokenizer.decode(ids))

[350, 2591, 402, 1688, 265, 745, 836, 319, 265, 7419]
The Supreme Court dismissed the appeal filed by the petitioner.


## 3. Build train / validation tokens

In [28]:
!pip install -q ijson

In [29]:
import ijson

### Source mix (share of tokens per court / document type)

In [30]:
SOURCE_MIX = {
    'Phase_1_Pretraining/Train/train_supreme_court.json': 0.40,
    'Phase_1_Pretraining/Train/train_acts.json':          0.30,
    'Phase_1_Pretraining/Train/train_high_courts.json':   0.20,
    'Phase_1_Pretraining/Train/train_tribunals.json':     0.10,
}

### Token buffers

In [31]:
# 'array' of unsigned 32-bit ints instead of Python lists (saves a lot of RAM)
train_tokens = array('I')
val_tokens   = array('I')
train_bounds, val_bounds = [], []        # cumulative end of each source (to label blocks by source)
EOS = tokenizer.eos_token_id

random.seed(SEED)

### Fast word-level encoder (cached; the tokenizer works word by word)

In [31]:
word_cache = {}                          # tokenizer works word by word, so caching per word is exact

def fast_encode(text):
    out = []
    for word in text.split():
        if word not in word_cache:
            word_cache[word] = tokenizer.encode(word)
        out.extend(word_cache[word])
    return out

### Stream the corpus, tokenize, fill train / validation

In [33]:
for fname, frac in SOURCE_MIX.items():
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type='dataset', token=HF_TOKEN)

    src_train_target = len(train_tokens) + int(TRAIN_TOKENS * frac)
    src_val_target   = len(val_tokens)   + int(VAL_TOKENS * frac)

    with open(path, 'rb') as f:                      # 'rb' for ijson
        for data in ijson.items(f, 'item'):          # streams one record at a time
            tokens = fast_encode(data['text'])
            tokens.append(EOS)                       # mark document boundary

            if random.random() < 0.05:
                if len(val_tokens) < src_val_target:
                    val_tokens.extend(tokens[:(src_val_target - len(val_tokens))])
            else:
                if len(train_tokens) < src_train_target:
                    train_tokens.extend(tokens[:(src_train_target - len(train_tokens))])

            if len(train_tokens) >= src_train_target and len(val_tokens) >= src_val_target:
                break

    train_bounds.append(len(train_tokens)); val_bounds.append(len(val_tokens))
    print(f'{fname.split("/")[-1]}: train so far {len(train_tokens):,}, val so far {len(val_tokens):,}')
    gc.collect()

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

train_supreme_court.json: train so far 160,000,000, val so far 4,000,000


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

...

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

train_high_courts.json: train so far 360,000,000, val so far 9,000,000


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

train_tribunals.json: train so far 400,000,000, val so far 10,000,000


### Totals

In [34]:
print(f'Train tokens: {len(train_tokens):,}')
print(f'Validation tokens: {len(val_tokens):,}')

Train tokens: 400,000,000
Validation tokens: 10,000,000


## 4. Token blocks

In [35]:
def create_blocks(tokens, bounds, block_size):
    assert tokens.itemsize == 4
    arr = np.frombuffer(tokens, dtype=np.uint32)
    n = len(arr) // block_size
    blocks = arr[:n * block_size].reshape(n, block_size)             # (n_blocks, block_size), no copy
    starts = np.arange(n) * block_size
    src = np.minimum(np.searchsorted(bounds, starts, side='right'), len(bounds) - 1)  # 0=SC 1=Acts 2=HC 3=Tribunals
    return blocks, src.astype(np.int64)

In [36]:
train_blocks, train_src = create_blocks(train_tokens, train_bounds, TRAIN_BLOCK_SIZE)
val_blocks,   val_src   = create_blocks(val_tokens,   val_bounds,   TRAIN_BLOCK_SIZE)
print('Training sequences:', len(train_blocks), '| Validation sequences:', len(val_blocks))

Training sequences: 195312 | Validation sequences: 4882


In [37]:
def get_batch(blocks, srcs, idx):
    chunk = torch.from_numpy(blocks[idx].astype(np.int64)).to(device)
    x, y = chunk[:, :-1], chunk[:, 1:]
    s = None if srcs is None else torch.from_numpy(srcs[idx]).to(device)
    return x, y, s

## Model code (hand-written)

In [12]:
import math
from dataclasses import dataclass
from typing import List, Optional
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

### Config

In [13]:
@dataclass
class Config:
    vocab_size: int = 32000
    d_model: int = 768
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: Optional[int] = None      # None -> MHA; < n_head -> GQA
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    rope_pct: float = 1.0                # fraction of the head dim that is rotated (partial RoPE)
    dropout: float = 0.0
    tie_embeddings: bool = False
    qk_norm: bool = False
    grad_ckpt: bool = False
    value_residual: bool = False         # later layers mix in the first layer's values
    mtp_heads: int = 0                   # extra multi-token-prediction heads (t+2, t+3, ...)
    mtp_weight: float = 0.3
    n_sources: int = 0                   # >0 -> source embedding (SC/Acts/HC/Tribunal) + 1 "unknown"
    src_drop: float = 0.3                # prob. of replacing the source id by "unknown" in training

    @property
    def head_dim(self):
        return self.d_model // self.n_head

    @property
    def kv_heads(self):
        return self.n_kv_head or self.n_head

    @property
    def ff_dim(self):                    # SwiGLU hidden size, parameter-matched to a 4*d GELU MLP
        return int(round(8 / 3 * self.d_model / 64)) * 64

### Building blocks: RMSNorm, RoPE, SwiGLU MLP

In [14]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.w.float()).type_as(x)


class Rotary(nn.Module):
    def __init__(self, dim, max_len, theta):
        super().__init__()
        inv = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / max(dim, 1)))
        freqs = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, T, start=0):
        """cos/sin for positions start .. start+T-1 (start > 0 only while generating with a cache)."""
        return self.cos[start:start + T], self.sin[start:start + T]


def apply_rope(x, cos, sin, rot_dim):
    """x: (B,H,T,hd). Rotates the first rot_dim dims (half-split convention)."""
    if rot_dim == 0:
        return x
    xr, xp = x[..., :rot_dim], x[..., rot_dim:]
    x1, x2 = xr.float().chunk(2, dim=-1)
    c, s = cos[None, None], sin[None, None]
    xr = torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)
    return torch.cat([xr, xp], dim=-1)


class MLP(nn.Module):
    """SwiGLU feed-forward."""
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg.d_model, cfg.ff_dim
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

### KV cache (new: lets generation process one token per step instead of the whole prefix)

In [15]:
# --------------------------------------------------------------------------------------
# KV cache + attention
# --------------------------------------------------------------------------------------
class KVCache:
    """Pre-allocated key/value buffer for ONE attention layer (stored un-expanded, i.e. Hkv heads).
    Lets generation process one new token per step instead of re-running the whole prefix."""

    def __init__(self, max_len):
        self.max_len, self.n, self.k, self.v = max_len, 0, None, None

    def append(self, k, v):
        if self.k is None:
            B, H, _, hd = k.shape
            self.k = k.new_empty(B, H, self.max_len, hd)
            self.v = v.new_empty(B, H, self.max_len, hd)
        T = k.size(2)
        self.k[:, :, self.n:self.n + T] = k
        self.v[:, :, self.n:self.n + T] = v
        self.n += T

    def get(self, window=None):
        s = 0 if window is None else max(0, self.n - window)
        return self.k[:, :, s:self.n], self.v[:, :, s:self.n]


def sdpa_gqa(q, k, v, mask=None, causal=False, p=0.0):
    """q: (B,H,T,hd)   k,v: (B,Hkv,S,hd).  Handles GQA without copying the cache when T == 1."""
    B, H, T, hd = q.shape
    Hkv = k.size(1)
    if T == 1 and mask is None:                       # single-token decode step
        if Hkv != H:                                  # fold the query-head group into the "length" axis
            r = H // Hkv
            out = F.scaled_dot_product_attention(q.reshape(B, Hkv, r, hd), k, v)
            return out.reshape(B, H, 1, hd)
        return F.scaled_dot_product_attention(q, k, v)
    if Hkv != H:
        r = H // Hkv
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p,
                                          is_causal=causal and mask is None)

### Self-attention (GQA / QK-norm / RoPE-or-NoPE / optional value residual, with KV-cache support)

In [16]:
class Attention(nn.Module):
    """MHA/GQA self-attention with optional RoPE (or NoPE), QK-norm, value residual, sliding-window mask
    and KV cache.  Returns (output, v_of_the_new_tokens)."""

    def __init__(self, cfg, causal=True, use_rope=True, value_residual=False):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout
        self.causal = causal
        self.rot_dim = (int(self.hd * cfg.rope_pct) // 2 * 2) if use_rope else 0
        self.lam = nn.Parameter(torch.zeros(self.Hkv, 1, 1)) if value_residual else None

    def forward(self, x, rope=None, v_first=None, mask=None, cache=None, window=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        if self.lam is not None and v_first is not None:
            lam = torch.sigmoid(self.lam).to(v.dtype)
            v = (1 - lam) * v + lam * v_first
        v_new = v
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        causal = self.causal
        if cache is not None:
            cache.append(k, v)
            k, v = cache.get(window if T == 1 else None)   # window only matters for 1-token steps
            causal = T > 1                                  # prefill: causal; decode step: sees everything cached
        out = sdpa_gqa(q, k, v, mask=mask, causal=causal, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, -1)), v_new

### LM head (final norm + tied unembedding + multi-token-prediction heads)

In [17]:
class LMHead(nn.Module):
    """Final norm + unembedding (optionally tied) + optional multi-token-prediction heads."""

    def __init__(self, cfg, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.norm = RMSNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.proj.weight = tok_emb.weight
        self.mtp = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False)
                                  for _ in range(getattr(cfg, "mtp_heads", 0))])

    def logits(self, h):
        return self.proj(self.norm(h))

    def forward(self, h, targets=None):
        h = self.norm(h)
        logits = self.proj(h)
        if targets is None:
            return logits, None
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1), ignore_index=-100)
        self.main_loss = loss.detach()          # next-token loss only (without MTP terms), for logging
        w = getattr(self.cfg, "mtp_weight", 0.3)
        for k, p in enumerate(self.mtp, start=1):
            if targets.size(1) <= k:
                break
            hk = h[:, :-k] + F.silu(p(h[:, :-k]))
            lk = self.proj(hk)
            loss = loss + w * F.cross_entropy(lk.reshape(-1, V).float(), targets[:, k:].reshape(-1),
                                              ignore_index=-100)
        return logits, loss

### Helpers: gradient checkpointing, weight init, base class

In [18]:
def ckpt_call(fn, enabled, *args):
    return checkpoint(fn, *args, use_reentrant=False) if enabled else fn(*args)


def init_weights(model, depth):
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    std = 0.02 / math.sqrt(2 * depth)
    for n, p in model.named_parameters():
        if n.endswith(("o.weight", "down.weight")):
            nn.init.normal_(p, 0.0, std)


class BaseLM(nn.Module):
    """Common interface used by training.py / generation.py:
         forward(x, targets=None, src=None) -> (logits, loss)
         prefill(prompt_ids, max_new)       -> (logits_for_next_token, state)      [generation]
         step(token_ids, state)             -> logits_for_next_token               [generation]
       `eval_offset` = number of leading label positions that are NOT scored (encoder-decoder only)."""
    eval_offset = 0

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        return n - self.tok.weight.numel() if non_embedding else n

### Transformer block

In [19]:
class Block(nn.Module):
    """Pre-norm residual block (attention + SwiGLU)."""

    def __init__(self, cfg, value_residual=False):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.attn = Attention(cfg, value_residual=value_residual)
        self.mlp = MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope, v_first=None, cache=None):
        h, v = self.attn(self.n1(x), rope, v_first, cache=cache)
        x = x + self.drop(h)
        return x + self.drop(self.mlp(self.n2(x))), v

### Decoder-only model

In [20]:
class DecoderLM(BaseLM):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.src_emb = nn.Embedding(cfg.n_sources + 1, cfg.d_model) if cfg.n_sources else None
        self.rope = Rotary(int(cfg.head_dim * cfg.rope_pct) // 2 * 2, cfg.max_seq_len, cfg.rope_theta)
        self.blocks = nn.ModuleList([Block(cfg, cfg.value_residual and i > 0) for i in range(cfg.n_layer)])
        self.head = LMHead(cfg, self.tok)
        init_weights(self, cfg.n_layer)

    def hidden(self, idx, src=None, caches=None, start=0):
        cfg = self.cfg
        B, T = idx.shape
        x = self.tok(idx)
        if self.src_emb is not None:
            unk = cfg.n_sources
            if src is None:                                  # unknown source (validation / test)
                src = torch.full((B,), unk, device=idx.device, dtype=torch.long)
            elif self.training and cfg.src_drop > 0:
                drop = torch.rand(B, device=idx.device) < cfg.src_drop
                src = torch.where(drop, torch.full_like(src, unk), src)
            x = x + self.src_emb(src)[:, None, :]
        rope = self.rope(T, start)
        v_first = None
        for i, blk in enumerate(self.blocks):
            if caches is None:
                x, v = ckpt_call(blk, cfg.grad_ckpt and self.training, x, rope, v_first)
            else:
                x, v = blk(x, rope, v_first, cache=caches[i])
            if v_first is None and cfg.value_residual:
                v_first = v
        return x

    def forward(self, idx, targets=None, src=None):
        return self.head(self.hidden(idx, src), targets)

    # ---- generation (KV cache) ----
    def prefill(self, idx, max_new):
        caches = [KVCache(idx.shape[1] + max_new) for _ in self.blocks]
        h = self.hidden(idx, caches=caches)
        return self.head.logits(h[:, -1]), caches

    def step(self, tok, caches):
        h = self.hidden(tok, caches=caches, start=caches[0].n)
        return self.head.logits(h[:, -1])

### Baseline / modified architecture definitions

In [21]:
Model = DecoderLM


BASE = dict(d_model=768, n_layer=12, n_head=12, max_seq_len=2048)


def cfg_baseline(**kw):
    return Config(**{**BASE, **kw})


def cfg_modified(**kw):
    return Config(**{**BASE, "n_layer": 17, "n_kv_head": 4, "qk_norm": True, "tie_embeddings": True,
                     "value_residual": True, "mtp_heads": 2, "n_sources": 4, **kw})

## 5. Build model

In [22]:
ARCH_NAME = 'arch1_decoder'
cfg_fn = {'baseline': cfg_baseline, 'modified': cfg_modified}[VARIANT]
cfg = cfg_fn(vocab_size=VOCAB_SIZE, max_seq_len=TRAIN_BLOCK_SIZE, grad_ckpt=GRAD_CKPT, **OVERRIDES)
model = Model(cfg).to(device)

n_params = model.num_params()
print(f'{ARCH_NAME} / {VARIANT}: {n_params/1e6:.1f}M parameters ({model.num_params(True)/1e6:.1f}M non-embedding)')
assert 50e6 <= n_params <= 300e6, 'parameter count must be between 50M and 300M'


arch1_decoder / modified: 145.0M parameters (108.2M non-embedding)


## 6. Train

In [49]:
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

### Training wrapper (forward + loss in one module)

In [50]:
class StepWrap(nn.Module):
    """Forward + loss in ONE module, so DataParallel only gathers scalars (never the huge logits tensor).
    Autocast has to live inside forward: DataParallel runs each GPU in its own thread."""
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x, y, src=None):
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
            if self.training:
                _, loss = self.m(x, y, src=src)
                return loss.reshape(1), self.m.head.main_loss.reshape(1)       # (loss to backprop, plain next-token loss to log)
            logits, _ = self.m(x)
        labels = y
        return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1),
                               reduction='sum').reshape(1)

### Model wrapper, optimizer, step counts

In [51]:
# `model` stays the plain module (checkpoints, generation); `wrap` is what we train / evaluate with.
wrap = nn.DataParallel(StepWrap(model)) if N_GPU > 1 else StepWrap(model)

decay = [p for p in model.parameters() if p.ndim == 2]
no_decay = [p for p in model.parameters() if p.ndim != 2]
optimizer = torch.optim.AdamW(
    [{'params': decay, 'weight_decay': WEIGHT_DECAY}, {'params': no_decay, 'weight_decay': 0.0}],
    lr=LR, betas=(0.9, 0.95), fused=(device == 'cuda'))
scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and amp_dtype == torch.float16))

seqs_per_step = MICRO_BATCH * GRAD_ACCUM
steps_per_epoch = len(train_blocks) // seqs_per_step
total_steps = steps_per_epoch * EPOCHS
print(f'{total_steps} optimizer steps, {seqs_per_step * (TRAIN_BLOCK_SIZE - 1):,} tokens per step, on {max(1, N_GPU)} GPU(s)')

1525 optimizer steps, 262,016 tokens per step, on 2 GPU(s)


### Learning-rate schedule

In [52]:
def lr_at(step, progress):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    return MIN_LR + 0.5 * (LR - MIN_LR) * (1 + math.cos(math.pi * min(1.0, progress)))

### Validation helpers

In [53]:
val_rng = np.random.RandomState(SEED)
quick_val_idx = np.sort(val_rng.choice(len(val_blocks), min(EVAL_BLOCKS, len(val_blocks)), replace=False))
all_val_idx = np.arange(len(val_blocks))

@torch.no_grad()
def evaluate(idx_list, bs=None):
    """Mean next-token cross-entropy on validation blocks (source unknown, like the test set)."""
    bs = bs or MICRO_BATCH
    wrap.eval()
    tot, cnt = 0.0, 0
    for i in range(0, len(idx_list), bs):
        x, y, _ = get_batch(val_blocks, None, idx_list[i:i + bs])
        tot += wrap(x, y).sum().item()
        cnt += y.size(0) * (y.size(1) - 0)
    wrap.train()
    return tot / cnt

### Checkpoint helpers

In [54]:
def save_ckpt(path, step, v):
    torch.save({'model': model.state_dict(), 'cfg': asdict(cfg), 'step': step, 'val_loss': v}, path)

best_path = os.path.join(MODEL_OUTPUT_DIR, 'best.pt')          # best validation checkpoint (model only)
resume_path = os.path.join(MODEL_OUTPUT_DIR, 'resume.pt')      # full training state, overwritten every SAVE_EVERY steps

def save_resume(step, elapsed):
    tmp = resume_path + '.tmp'                                  # write-then-rename: a crash can't corrupt the old file
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                'cfg': asdict(cfg), 'step': step, 'elapsed': elapsed, 'history': history, 'best_val': best_val}, tmp)
    os.replace(tmp, resume_path)

### Training state + resume from checkpoint

In [55]:
history = {'step': [], 'train_loss': [], 'val_step': [], 'val_loss': []}
best_val = float('inf')
step, elapsed_prev = 0, 0.0

# ---- resume if a checkpoint exists (in this session's output folder, or a resume.pt uploaded as a Kaggle dataset)
load_path = resume_path if os.path.exists(resume_path) else next(iter(glob.glob('/kaggle/input/**/resume.pt', recursive=True)), None)
if load_path:
    ck = torch.load(load_path, map_location=device, weights_only=False)
    assert ck['cfg'] == asdict(cfg), 'checkpoint was trained with a different model config (VARIANT / OVERRIDES)'
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); scaler.load_state_dict(ck['scaler'])
    step, elapsed_prev, history, best_val = ck['step'], ck['elapsed'], ck['history'], ck['best_val']
    del ck
    print(f'Resumed from {load_path} at step {step} ({elapsed_prev / 60:.1f} min already trained)')
else:
    print('No resume checkpoint found - training from scratch')

Resumed from /kaggle/input/datasets/kratikasum/model/resume.pt at step 750 (363.6 min already trained)


### Training loop

In [56]:
from tqdm import tqdm

t0 = time.time() - elapsed_prev            # training clock, includes time spent before a resume
t_run = time.time(); tok_seen = 0; run_loss = 0.0; run_n = 0
wrap.train()
done = False

# Initialize the global progress bar
pbar = tqdm(total=total_steps, initial=step, desc="Training",dynamic_ncols=True, position=0, leave=True)

for epoch in range(step // steps_per_epoch, EPOCHS):
    perm = np.random.RandomState(SEED + epoch).permutation(len(train_blocks))   # same order after a resume
    for s in range(steps_per_epoch):
        if epoch * steps_per_epoch + s < step:                  # already trained before the resume
            continue
        progress = max(step / total_steps, (time.time() - t0) / (MAX_TRAIN_MINUTES * 60))
        if progress >= 1.0:
            done = True; break
        lr = lr_at(step, progress)
        for g in optimizer.param_groups: g['lr'] = lr

        for m in range(GRAD_ACCUM):
            idx = perm[(s * GRAD_ACCUM + m) * MICRO_BATCH:(s * GRAD_ACCUM + m + 1) * MICRO_BATCH]
            x, y, src = get_batch(train_blocks, train_src, idx)
            loss, main = wrap(x, y, src=src)
            loss, main = loss.mean(), main.mean()                 # average over GPUs
            scaler.scale(loss / GRAD_ACCUM).backward()
            run_loss += main.item() / GRAD_ACCUM
            tok_seen += x.numel()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1; run_n += 1

        # Update the progress bar and show current metrics
        pbar.update(1)
        pbar.set_postfix({"loss": f"{run_loss / run_n:.4f}", "lr": f"{lr:.2e}"})

        if step % LOG_EVERY == 0:
            history['step'].append(step); history['train_loss'].append(run_loss / run_n)
            # Use pbar.write() instead of print() to avoid breaking the progress bar UI
            pbar.write(f'step {step}/{total_steps} | loss {run_loss / run_n:.4f} | lr {lr:.2e} | '
                  f'{tok_seen / (time.time() - t_run):,.0f} tok/s | {(time.time() - t0) / 60:.1f} min')
            run_loss = 0.0; run_n = 0
            
        if step % EVAL_EVERY == 0:
            v = evaluate(quick_val_idx)
            history['val_step'].append(step); history['val_loss'].append(v)
            pbar.write(f'   >> val loss {v:.4f} | ppl {math.exp(v):.2f}')
            if v < best_val:
                best_val = v
                save_ckpt(best_path, step, v)
                
        if step % SAVE_EVERY == 0:
            save_resume(step, time.time() - t0)
            
    if done:
        pbar.write('Time budget reached - stopping early.'); break

# Clean up the progress bar when done
pbar.close()

Training:  51%|█████     | 775/1525 [11:08<5:40:23, 27.23s/it, loss=2.6096, lr=3.13e-04]

step 775/1525 | loss 2.6096 | lr 3.13e-04 | 9,791 tok/s | 374.7 min


Training:  52%|█████▏    | 800/1525 [22:26<5:26:37, 27.03s/it, loss=2.5832, lr=3.00e-04]

step 800/1525 | loss 2.5832 | lr 3.00e-04 | 9,726 tok/s | 386.0 min


Training:  54%|█████▍    | 825/1525 [33:46<5:15:39, 27.06s/it, loss=2.5788, lr=2.87e-04]

...

step 1500/1525 | loss 2.3030 | lr 6.04e-05 | 9,505 tok/s | 708.2 min


Training:  98%|█████████▊| 1500/1525 [5:44:41<11:42, 28.08s/it, loss=2.3030, lr=6.04e-05]

   >> val loss 2.4218 | ppl 11.27


Training: 100%|██████████| 1525/1525 [5:56:27<00:00, 27.60s/it, loss=2.3059, lr=6.00e-05]

step 1525/1525 | loss 2.3059 | lr 6.00e-05 | 9,495 tok/s | 720.0 min


### Final checkpoint + loss plot

In [57]:
# final checkpoint comparison on the quick val set, keep the best one
v = evaluate(quick_val_idx)
history['val_step'].append(step); history['val_loss'].append(v)
save_ckpt(os.path.join(MODEL_OUTPUT_DIR, 'last.pt'), step, v)
if v < best_val or not os.path.exists(best_path):
    best_val = min(best_val, v)
    save_ckpt(best_path, step, v)
save_resume(step, time.time() - t0)
train_minutes = (time.time() - t0) / 60
print(f'Done: {step} steps in {train_minutes:.1f} min | best quick-val loss {best_val:.4f}')

Done: 1525 steps in 720.2 min | best quick-val loss 2.4192


In [58]:
plt.figure(figsize=(7, 4))
plt.plot(history['step'], history['train_loss'], label='train')
plt.plot(history['val_step'], history['val_loss'], 'o-', label='validation')
plt.xlabel('optimizer step'); plt.ylabel('next-token loss'); plt.legend(); plt.grid(alpha=.3)
plt.title(f'{ARCH_NAME} / {VARIANT}')
plt.savefig(LOSS_PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()

## 7. Best checkpoint: validation perplexity

In [59]:
# ---- reload best checkpoint and measure it on the FULL validation set
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
full_val = evaluate(all_val_idx)
print(f'Best checkpoint (step {ckpt["step"]}): val loss {full_val:.4f} | perplexity {math.exp(full_val):.2f}')
json.dump({'arch': ARCH_NAME, 'variant': VARIANT, 'overrides': OVERRIDES, 'params_M': n_params / 1e6,
           'best_step': ckpt['step'], 'val_loss': full_val, 'val_ppl': math.exp(full_val),
           'train_minutes': train_minutes, 'history': history},
          open(os.path.join(OUTPUT_DIR, 'results.json'), 'w'), indent=1)

Best checkpoint (step 1525): val loss 2.3423 | perplexity 10.40


## 8. Generate predictions and create submission zip

### Optional: generate from a saved checkpoint (skip if you just trained above)

In [23]:
# OPTIONAL - generate without training: point CKPT at a saved best.pt / last.pt, then run this cell.
# (Needed cells: 1 Configuration, 2 tokenizer, "fast_encode", Model code, 5 Build model, then this section.)
CKPT = '/kaggle/input/datasets/sunnysfhbv/model-arch1/last.pt'     # e.g. '/kaggle/input/my-run/legal-final/best.pt'
if CKPT:
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); model.eval()
    print('loaded', CKPT, '(step', ck['step'], ')')

loaded /kaggle/input/datasets/sunnysfhbv/model-arch1/last.pt (step 1525 )


### Sampling (top-p is applied on the top-k candidates only)

In [24]:
def sample_logits(logits, prev=None, temperature=0.8, top_k=50, top_p=0.95,
                  repetition_penalty=1.0, no_repeat_ngram=3):
    logits = logits.float()

    # ---- NEW: n-gram blocking ----
    if no_repeat_ngram > 0 and prev is not None and prev.size(1) >= no_repeat_ngram:
        B = logits.size(0)
        for b in range(B):
            gen = prev[b].tolist()
            ngram_suffix = tuple(gen[-(no_repeat_ngram - 1):])
            banned = set()
            for i in range(len(gen) - no_repeat_ngram + 1):
                if tuple(gen[i:i + no_repeat_ngram - 1]) == ngram_suffix:
                    banned.add(gen[i + no_repeat_ngram - 1])
            if banned:
                logits[b, list(banned)] = float("-inf")

    if repetition_penalty != 1.0 and prev is not None:
        s = torch.gather(logits, 1, prev)
        s = torch.where(s < 0, s * repetition_penalty, s / repetition_penalty)
        logits = logits.scatter(1, prev, s)
    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k < logits.size(-1):
        vals, inds = torch.topk(logits, top_k)
        if top_p < 1.0:
            probs = F.softmax(vals, -1)
            vals = vals.masked_fill((probs.cumsum(-1) - probs) > top_p, float("-inf"))
        choice = torch.multinomial(F.softmax(vals, -1), 1)
        return inds.gather(1, choice)
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True)
        probs = F.softmax(sl, -1)
        sl = sl.masked_fill((probs.cumsum(-1) - probs) > top_p, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
    return torch.multinomial(F.softmax(logits, -1), 1)

### Word-end token mask + fast generate() (prefill once, then one token per step)

In [25]:
# bool[V]: True for tokens that end a word. Counting them = counting words, without decoding text every step.
word_end = torch.tensor([tokenizer.inverse_vocab.get(i, b'').endswith(tokenizer.word_end_marker)
                         for i in range(VOCAB_SIZE)], device=device)

In [26]:
@torch.inference_mode()
def generate(model, idx, max_new_tokens=200, temperature=0.8, top_k=50, top_p=0.95,
             repetition_penalty=1.0, ban_ids=None, target_words=None, check_every=8):
    """idx: (B, L) equal-length prompts -> prompt + continuation, shape (B, L+new).
    The prompt goes through the model ONCE (prefill); after that every step feeds a single token."""
    was_training = model.training
    model.eval()
    logits, state = model.prefill(idx, max_new_tokens)
    gen = idx
    prompt_len = idx.size(1)                                # CHANGED: remember where the prompt ends
    words = torch.zeros(idx.size(0), device=idx.device, dtype=torch.long)
    for i in range(max_new_tokens):
        if ban_ids:                                         # never emit pad/unk/bos/eos
            logits[:, ban_ids] = float("-inf")
        prev = gen[:, prompt_len:] if i > 0 else None       # CHANGED: penalize only generated tokens
        nxt = sample_logits(logits, prev, temperature, top_k, top_p, repetition_penalty)
        gen = torch.cat([gen, nxt], dim=1)
        if target_words is not None:
            words += word_end[nxt.squeeze(1)].long()
            if (i + 1) % check_every == 0 and bool((words >= target_words).all()):
                break
        if i + 1 < max_new_tokens:
            logits = model.step(nxt, state)                 # ONE token per step
    model.train(was_training)
    return gen

In [27]:
# !pip install -q bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.6 MB/s eta 0:00:00


In [38]:
# from bert_score import BERTScorer
# # scorer = BERTScorer(lang='en', batch_size=64)   # loads ONCE; add model_type='...' to match the grader
# scorer = BERTScorer(model_type='roberta-large', lang='en', batch_size=64)

# import re

# def lcs_length(a, b):
#     """Mirrors the eval script's LCS exactly."""
#     if not a or not b: return 0
#     if len(a) < len(b): a, b = b, a
#     masks = {}
#     for j, t in enumerate(a): masks[t] = masks.get(t, 0) | (1 << j)
#     state = 0
#     for t in b:
#         m = masks.get(t, 0); x = m | state; y = (state << 1) | 1; state = x & ~(x - y)
#     return bin(state).count('1')

# def rouge_l_f1(p, r):
#     """Mirrors the eval script's ROUGE-L F1 exactly."""
#     pt, rt = p.split(), r.split()
#     if not pt or not rt: return 0.0
#     l = lcs_length(pt, rt)
#     if not l: return 0.0
#     pp = l / len(pt); rr = l / len(rt)
#     return 0.0 if pp + rr == 0 else 2 * pp * rr / (pp + rr)

# def clean_generation(text, max_words=128):
#     text = ' '.join(text.strip().split())
#     # Trim to complete sentences within word limit
#     words = text.split()[:max_words]
#     text = ' '.join(words)
#     # Find last sentence-ending punctuation
#     match = list(re.finditer(r'[.!?]', text))
#     if match:
#         last_end = match[-1].end()
#         trailing = text[last_end:].strip()
#         if trailing:  # ANY trailing fragment → trim it (was >3 words)
#             text = text[:last_end].strip()
#     return text


# @torch.inference_mode()
# def mbr_generate(x, n=32, temperature=0.5, top_k=30, top_p=0.92, rep=1.05):
#     B = x.size(0)
#     xr = x.repeat_interleave(n, dim=0)
#     with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
#         out = generate(model, xr, max_new_tokens=MAX_GENERATION_TOKENS, temperature=temperature,
#                        top_k=top_k, top_p=top_p, repetition_penalty=rep, ban_ids=ban_ids,
#                        target_words=TARGET_GENERATION_WORDS + 4)
#     # Trim each candidate to TARGET words (will be trimmed to 128 later after selection)
#     texts = [' '.join(tokenizer.decode(r.tolist()).split()[:TARGET_GENERATION_WORDS])
#              for r in out[:, x.size(1):]]

#     # ---- BERTScore pairwise (same approach, batched) ----
#     pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
#     cands, refs_ = [], []
#     for b in range(B):
#         c = texts[b * n:(b + 1) * n]
#         for i, j in pairs:
#             cands.append(c[i]); refs_.append(c[j])
#     f_bert = scorer.score(cands, refs_)[2].view(B, len(pairs))

#     best = []
#     for b in range(B):
#         candidates = texts[b * n:(b + 1) * n]

#         # BERTScore pairwise matrix
#         M_bert = torch.zeros(n, n)
#         for k, (i, j) in enumerate(pairs):
#             M_bert[i, j] = M_bert[j, i] = f_bert[b, k]
#         bert_scores = M_bert.sum(1)

#         # ROUGE-L pairwise scores (mirrors the eval script exactly)
#         rouge_scores = torch.zeros(n)
#         for i in range(n):
#             for j in range(n):
#                 if i != j:
#                     rouge_scores[i] += rouge_l_f1(candidates[i], candidates[j])

#         # Combined: BERTScore is the primary metric, ROUGE-L helps break ties
#         bs_norm = bert_scores / bert_scores.max().clamp(min=1e-8)
#         rl_norm = rouge_scores / rouge_scores.max().clamp(min=1e-8)
#         combined = 0.9 * bs_norm + 0.1 * rl_norm

#         winner = candidates[int(combined.argmax())]
#         best.append(clean_generation(winner, max_words=128))   # post-process + hard trim to 128
#     return best

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Load test prompts

In [39]:
# ---- load test prompts
test_path = None
for fname in TEST_FILES:
    try:
        test_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type='dataset', token=HF_TOKEN)
        break
    except Exception as e:
        print('could not get', fname)
test = json.load(open(test_path, encoding='utf-8'))
if isinstance(test, list):
    test = {str(i + 1): r for i, r in enumerate(test)}
keys = list(test.keys())
print(len(keys), 'test prompts')

1000 test prompts


### Tokenize prompts, sort by length

In [40]:
prompt_ids = {k: fast_encode(test[k]['text']) for k in keys}
order = sorted(keys, key=lambda k: len(prompt_ids[k]))      # similar lengths together -> no padding needed
ban_ids = [tokenizer.vocab[t] for t in tokenizer.reserved_tokens if t in tokenizer.vocab]
MAX_PROMPT = cfg.max_seq_len - MAX_GENERATION_TOKENS         # prompt + continuation must fit the RoPE table
print('longest prompt:', max(len(v) for v in prompt_ids.values()), 'tokens | kept at most', MAX_PROMPT)

longest prompt: 941 tokens | kept at most 1664


### Generate (progress + ETA printed per batch)

In [41]:
predictions = {}
t0 = time.time()
MBR_BATCH = 1          # halved: each prompt now gets n=16 samples (2*16=32 sequences per batch)
for b in range(0, len(order), MBR_BATCH):
    batch_keys = order[b:b + MBR_BATCH]
    L = min(min(len(prompt_ids[k]) for k in batch_keys), MAX_PROMPT)
    x = torch.tensor([prompt_ids[k][-L:] for k in batch_keys], device=device)
    best = mbr_generate(x)    # uses new defaults: n=16, temp=0.4, top_k=20, top_p=0.88, rep=1.08
    for k, text in zip(batch_keys, best):
        predictions[k] = {'text': text}
    done = min(b + MBR_BATCH, len(order)); el = time.time() - t0
    print(f'{done}/{len(order)} done | {el / 60:.1f} min | ETA {el / done * (len(order) - done) / 60:.1f} min')

1/1000 done | 0.1 min | ETA 96.8 min
2/1000 done | 0.2 min | ETA 103.3 min
3/1000 done | 0.3 min | ETA 108.0 min
4/1000 done | 0.4 min | ETA 104.1 min
5/1000 done | 0.5 min | ETA 100.6 min
6/1000 done | 0.6 min | ETA 101.2 min
7/1000 done | 0.7 min | ETA 103.0 min
8/1000 done | 0.8 min | ETA 103.4 min
9/1000 done | 0.9 min | ETA 103.5 min
10/1000 done | 1.0 min | ETA 103.7 min
11/1000 done | 1.1 min | ETA 102.1 min
12/1000 done | 1.3 min | ETA 103.2 min
13/1000 done | 1.3 min | ETA 102.3 min
14/1000 done | 1.4 min | ETA 101.5 min
15/1000 done | 1.6 min | ETA 102.2 min
16/1000 done | 1.7 min | ETA 101.8 min
17/1000 done | 1.8 min | ETA 101.7 min
18/1000 done | 1.9 min | ETA 101.1 min
19/1000 done | 2.0 min | ETA 101.3 min
20/1000 done | 2.1 min | ETA 101.5 min
21/1000 done | 2.2 min | ETA 101.5 min
22/1000 done | 2.3 min | ETA 101.6 min
23/1000 done | 2.4 min | ETA 101.1 min
24/1000 done | 2.5 min | ETA 100.9 min
25/1000 done | 2.6 min | ETA 100.6 min
26/1000 done | 2.7 min | ETA 100.8 

In [66]:
predictions = {}
t0 = time.time()
for b in range(0, len(order), GEN_BATCH):
    batch_keys = order[b:b + GEN_BATCH]
    L = min(min(len(prompt_ids[k]) for k in batch_keys), MAX_PROMPT)   # keep the last L tokens of every prompt
    x = torch.tensor([prompt_ids[k][-L:] for k in batch_keys], device=device)
    with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
        out = generate(model, x, max_new_tokens=MAX_GENERATION_TOKENS, temperature=TEMPERATURE, top_k=TOP_K,
                       top_p=TOP_P, repetition_penalty=REPETITION_PENALTY, ban_ids=ban_ids,
                       target_words=TARGET_GENERATION_WORDS + 2)       # +2 words of margin, trimmed below
    for k, row in zip(batch_keys, out[:, L:]):
        words = tokenizer.decode(row.tolist()).split()[:TARGET_GENERATION_WORDS]
        predictions[k] = {'text': ' '.join(words)}
    done = min(b + GEN_BATCH, len(order)); el = time.time() - t0
    print(f'{done}/{len(order)} done | {el / 60:.1f} min | ETA {el / done * (len(order) - done) / 60:.1f} min')

32/1000 done | 0.1 min | ETA 2.3 min
64/1000 done | 0.2 min | ETA 2.5 min
96/1000 done | 0.3 min | ETA 2.4 min
128/1000 done | 0.3 min | ETA 2.3 min
160/1000 done | 0.4 min | ETA 2.2 min
192/1000 done | 0.5 min | ETA 2.1 min
224/1000 done | 0.6 min | ETA 2.0 min
256/1000 done | 0.7 min | ETA 1.9 min
288/1000 done | 0.8 min | ETA 1.9 min
320/1000 done | 0.9 min | ETA 1.8 min
352/1000 done | 0.9 min | ETA 1.7 min
384/1000 done | 1.0 min | ETA 1.6 min
416/1000 done | 1.1 min | ETA 1.6 min
448/1000 done | 1.2 min | ETA 1.5 min
480/1000 done | 1.3 min | ETA 1.4 min
512/1000 done | 1.4 min | ETA 1.3 min
544/1000 done | 1.5 min | ETA 1.2 min
576/1000 done | 1.5 min | ETA 1.1 min
608/1000 done | 1.6 min | ETA 1.0 min
640/1000 done | 1.7 min | ETA 1.0 min
672/1000 done | 1.8 min | ETA 0.9 min
704/1000 done | 2.0 min | ETA 0.8 min
736/1000 done | 2.1 min | ETA 0.8 min
768/1000 done | 2.2 min | ETA 0.7 min
800/1000 done | 2.3 min | ETA 0.6 min
832/1000 done | 2.4 min | ETA 0.5 min
864/1000 done |

### Write prediction.json and submission.zip

In [42]:
predictions = {k: predictions[k] for k in keys}               # original order
with open(PREDICTION_PATH, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=1)
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PREDICTION_PATH, arcname='prediction.json')
print('Saved', PREDICTION_PATH, 'and', SUBMISSION_ZIP)
print('\nExample continuation:\n', predictions[keys[0]]['text'])

Saved /kaggle/working/arch1_decoder_modified/prediction.json and /kaggle/working/arch1_decoder_modified/submission.zip

Example continuation:
 well settled that the High Court should not entertain a writ petition after a lapse of a long time. The delay is not the cause of action for the writ petition and the writ petition, therefore, must be dismissed on this ground alone. 11. In view of the law as laid down by this Court in the cases referred to above, it is evident that the acquisition proceedings must be set at naught if there is no legal impediment in taking the decision to challenge the notification under Section 4 and declaration under Section 6 of the Act. In view thereof, the writ petitions are dismissed. However, in the facts and circumstances of the case, there shall be no order as to costs. 12.


In [69]:
import os

target = "arch1_decoder_modified"

for root, dirs, files in os.walk("/kaggle"):
    if os.path.basename(root) == target:
        print(f"FOUND: {root}\n")
        
        for r, d, f in os.walk(root):
            for file in f:
                print(os.path.join(r, file))


FOUND: /kaggle/working/arch1_decoder_modified

/kaggle/working/arch1_decoder_modified/results.json
/kaggle/working/arch1_decoder_modified/prediction.json
/kaggle/working/arch1_decoder_modified/training_validation_loss.png
/kaggle/working/arch1_decoder_modified/submission.zip
/kaggle/working/arch1_decoder_modified/legal-final/best.pt
/kaggle/working/arch1_decoder_modified/legal-final/resume.pt
/kaggle/working/arch1_decoder_modified/legal-final/last.pt


In [71]:
import shutil
import os

source = "/kaggle/working/arch1_decoder_modified"
output = "/kaggle/working/arch1_decoder_modified_all"

shutil.make_archive(output, "zip", source)

zip_path = output + ".zip"

print(f"Created: {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / (1024**3):.2f} GB")


Created: /kaggle/working/arch1_decoder_modified_all.zip
Size: 2.49 GB
